# Ingest data using Auto Loader and Structured Streaming

In [0]:
CATALOG_NAME = "dbr_dev_ua5816bd"
STORAGE_ACCOUNT = "dlsua5816bd"
LOGIN = "oles0305"

adls_url = f'abfss://{LOGIN}@{STORAGE_ACCOUNT}.dfs.core.windows.net'

source_path = f"{adls_url}/raw_data/raw_results_data"
schema_location = f"{adls_url}/raw_data/_schema"

ingestion_target_path = f"{adls_url}/raw_data/results_ingested/results_files"
checkpoint_location = f"{adls_url}/raw_data/results_ingested/_checkpoint"

bronze_table = f"{CATALOG_NAME}.{LOGIN}_bronze.results"

### Configuring Auto Loader

In [0]:
df = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.maxFilesPerTrigger", 300)
        .load(source_path)
)

### Enriching Bronze Stream with Metadata

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_bronze = (
    df.withColumn('ingested_at', current_timestamp())
        .withColumn('file_path', col("_metadata.file_path"))
)

### Write the streaming DataFrame to the Bronze Delta table

In [0]:
write_query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .trigger(availableNow=True)
        .table(bronze_table)
)

write_query.awaitTermination()

### Streaming Query Statistics
- **Input Volume:** 1000 files
- **Batch Config:** `cloudFiles.maxFilesPerTrigger = 300`
- **Execution:** 4 micro-batches (300 + 300 + 300 + 100 files)


![screen_1_1.png](./screenshots/screen_1_1.png "screen_1_1.png")

#####Verify that the initial batch was successfully written to Delta Lake and check sample records with metadata columns (`ingested_at`, `file_path`).

In [0]:
%sql
SELECT * 
FROM dbr_dev_ua5816bd.oles0305_bronze.results 
LIMIT 10

# Experiments

###Experiment 1: Schema Evolution (Adding a New Column)

In [0]:
extra_col = [{
    "resultId": 30000,
    "driverId": 1000,
    "extra_col_value": 777
}]

(
    spark.createDataFrame(extra_col)
        .write
        .mode("append")
        .json(f"{source_path}/extra_row_sample.json")
)

In [0]:
write_extra_query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table(bronze_table)
)

write_extra_query.awaitTermination()

![](./screenshots/screen_1_5.png)

`extra_col_value` has been appended to the Delta table schema, while previous records maintain `NULL` for this attribute

### Experiment 2: Corrupted Record and `_rescued_data` column

In [0]:
corrupted_row = [{
    "resultId": 30001,
    "driverId": 1001,
    "points": "must be double"
}]

(
    spark.createDataFrame(corrupted_row)
        .write
        .mode("append")
        .json(f"{source_path}/corrupted_row_sample.json")
)

In [0]:
write_corrupted_query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table(bronze_table)
)

write_corrupted_query.awaitTermination()

![](./screenshots/screen_1_3.png "screen_1_3.png")

The stream does not fail. Instead, the invalid payload is captured safely inside the `_rescued_data` column, preserving data for debugging.

### Experiment 3: Safe Reload and Idempotency Check

In [0]:
initial_count = spark.table(bronze_table).count()

write_empty_query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_location)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table(bronze_table)
)

write_empty_query.awaitTermination()

In [0]:
print(f"Number of rows in the table:\nBefore: {initial_count}\nAfter: {spark.table(bronze_table).count()}")

![screen_1_4.png](./screenshots/screen_1_4.png "screen_1_4.png")